# Lacunae Analysis Package Walkthrough

This notebook keeps the exploratory thresholding, segmentation, and density review workflow while importing the shared package APIs from `src/lacunae_analysis/`.


## Environment

Install the package in your active environment before running the notebook:

```bash
pip install -e .
```


In [ ]:
%matplotlib inline

from pathlib import Path

import pandas as pd

from lacunae_analysis.io import load_scan
from lacunae_analysis.metrics import analyze_lacuna_density
from lacunae_analysis.models import DensityFilterSettings, ScanInput, ThresholdSettings
from lacunae_analysis.pipeline import run_single_scan, run_single_scan_job
from lacunae_analysis.segmentation import plot_lacuna_segmentation, segment_lacunae
from lacunae_analysis.thresholding import compute_threshold


## Choose Inputs

Point `scan_path` at a local `.aim` file and choose an output directory for persisted figures and summary tables.


In [ ]:
scan_path = Path("../path/to/scan.aim")
output_dir = Path("../outputs/notebook-walkthrough")
output_dir.mkdir(parents=True, exist_ok=True)
intensity_unit = "bmd"

if not scan_path.exists():
    raise FileNotFoundError("Update `scan_path` to point at a local .aim scan before running the walkthrough.")

scan_path


## Load the Scan

Use the package loader so the notebook sees the same selected intensity image used by the CLI and tests. AIM defaults to BMD-calibrated values; set `intensity_unit = "raw"` for native greyscale or `"hu"` for Hounsfield units.


In [ ]:
loaded_scan = load_scan(scan_path, intensity_unit=intensity_unit)

pd.Series(
    {
        "source_path": str(loaded_scan.source_path),
        "shape": loaded_scan.voxel_data.shape,
        "spacing": loaded_scan.spacing,
        "origin": loaded_scan.origin,
        "intensity_unit": intensity_unit,
        "units": loaded_scan.units,
        "density_slope": loaded_scan.density_slope,
        "density_intercept": loaded_scan.density_intercept,
    }
)


## Inspect Threshold Selection

`compute_threshold` preserves the notebook-friendly histogram workflow while reusing the shared thresholding module.


In [ ]:
threshold_results = compute_threshold(loaded_scan)
pd.Series(threshold_results)


## Segment Lacunae

Run the shared segmentation step and keep the exploratory overlay plot in the notebook.


In [ ]:
segmentation_results = segment_lacunae(
    loaded_scan,
    threshold=float(threshold_results["selected_threshold"]),
    bone_sigma=10.0,
    lacuna_sigma=1.2,
)

plot_lacuna_segmentation(segmentation_results)


## Measure Lacuna Density

Use the metrics module directly when you want to inspect the component table before running the full pipeline.


In [ ]:
density_filter_settings = DensityFilterSettings()
density_results = analyze_lacuna_density(
    segmentation_results["lacuna_binary_sitk"],
    segmentation_results["bone_mask_sitk"],
    lower_volume_um3=density_filter_settings.lower_volume_um3,
    upper_volume_um3=density_filter_settings.upper_volume_um3,
    include_edge_lacunae=density_filter_settings.include_edge_lacunae,
    edge_width=density_filter_settings.edge_width,
)

pd.Series(density_results["summary"])


In [ ]:
density_results["component_table"].head()


## Run the Shared Single-Scan Pipeline

This reproduces the end-to-end package path used by the CLI while keeping the intermediate objects available for notebook exploration.


In [ ]:
pipeline_results = run_single_scan(
    ScanInput(image_path=scan_path, output_dir=output_dir, intensity_unit=intensity_unit),
    threshold_settings=ThresholdSettings(),
    density_filter_settings=DensityFilterSettings(),
)

pd.Series(pipeline_results["summary"])


## Persist Notebook Outputs

Use `run_single_scan_job` when you want the same deterministic figures and summary files that the CLI writes to disk.


In [ ]:
job_results = run_single_scan_job(
    scan_path,
    config={"scan": {"intensity_unit": intensity_unit}},
    output_dir=output_dir,
)

pd.Series(job_results["summary"])


In [ ]:
sorted(path.name for path in output_dir.iterdir())
